In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 79361)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 35979)

# #### Device() ####
# device = cuda:6

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:6)
# edge_attr                (32464, 16)              Tensor (cuda:6)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:6)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [2]:
from modules.layers import AttentionSetPooling
from modules.loss import MultiLoss, NBLoss
from modules.model import BaseAutoencoder, CountsAutoencoder
from modules.norm import LogCounts
from modules.train import Loader
from modules.trainers import ReconstrTrainer
from modules.utils import dict_summary
import torch
import torch.nn as nn

In [3]:
loader = Loader(
    dataset,
    device=device,
    batch_size=128,
)

ae = CountsAutoencoder(
    dataset=dataset,
    embed_dim=128,
    method='node',

    norm_class=LogCounts,
    encoder_class=nn.Linear,
    pooling_class=AttentionSetPooling,
    variational=False,

    hidden_dims=1,
    act_fn=nn.ReLU,
    norm_fn='layer',

    norm_kwargs={
        'libnorm':True,
        'znorm':True,
        'learnable':True
    }
)

trainer = ReconstrTrainer(
    lr=1e-3, 
    trainer_norm_class=LogCounts,
    early_stop=True,
    stop_metric='nb',
    
    kw_keys={'x':'x', 'mu':'x_pred', 'theta':'theta'},
    # kw_keys=('x_t_pred', 'x_t', 'x_pred', 'x', 'theta'),
    metric_keys={'pred':'mu', 'target':'x'},
    
    loss_class=NBLoss,
    # loss_class=MultiLoss,
    # loss_kwargs={
    #     'loss_classes': [nn.MSELoss, NBLoss],
    #     'loss_weights': [0.0, 1.0],
    #     'loss_inputs':[
    #         {'input':'x_t_pred', 'target':'x_t'},
    #         {'x':'x', 'mu':'x_pred', 'theta':'theta'}
    #     ],
    # },

    

)

In [4]:
trainer.run(
    model=ae,
    loader=loader,
    num_epochs=300,
    report_metrics=['loss','kld','nb','mae'],
    verbose=True
)

  0%|          | 0/300 [00:00<?, ?it/s]

Test	 loss=7.8992    nb=7.9048    mae=0.5317



In [5]:
out = trainer.model(_batch, need_weights=False)
if isinstance(out, torch.Tensor):
    print(out.shape)
elif isinstance(out, dict):
    print(dict_summary(out))
else:
    print(out)

# x                        (64, 4373)               Tensor (cuda:6)
# x_t                      (64, 4373)               Tensor (cuda:6)
# libsize                  (64,)                    Tensor (cuda:6)
# libscale                 ()                       Tensor (cuda:6)
# batch_size               64                       int
# num_nodes                4373                     int
# z_mu                     NoneType
# z_logvar                 NoneType
# x_t_pred                 (64, 4373)               Tensor (cuda:6)
# x_pred                   (64, 4373)               Tensor (cuda:6)
# theta                    (1, 4373)                Tensor (cuda:6)



In [6]:
trainer.dev_metrics

{0: {'train': {'loss': 9.577655383518763,
   'mse': 0.8429387211799622,
   'rmse': 0.9181169271469116,
   'mae': 0.6236953139305115,
   'r2': 0.7796393632888794,
   'nb': 9.667069435119629},
  'val': {'loss': 8.6443510055542,
   'mse': 0.8705331683158875,
   'rmse': 0.9330236911773682,
   'mae': 0.6338615417480469,
   'r2': 0.7729129195213318,
   'nb': 8.584171295166016}},
 1: {'train': {'loss': 8.725064550127302,
   'mse': 0.8877091407775879,
   'rmse': 0.9421831965446472,
   'mae': 0.6332557797431946,
   'r2': 0.7679355144500732,
   'nb': 8.597626686096191},
  'val': {'loss': 8.393575191497803,
   'mse': 0.9685212969779968,
   'rmse': 0.9841347932815552,
   'mae': 0.6606638431549072,
   'r2': 0.7473517656326294,
   'nb': 8.356232643127441}},
 2: {'train': {'loss': 8.373261043003627,
   'mse': 0.9915003776550293,
   'rmse': 0.9957411289215088,
   'mae': 0.659920334815979,
   'r2': 0.7408024668693542,
   'nb': 8.390379905700684},
  'val': {'loss': 8.293559074401855,
   'mse': 1.0822240

In [7]:
trainer.test_metrics

{'loss': 7.899179458618164,
 'mse': 0.7447174191474915,
 'rmse': 0.8629701137542725,
 'mae': 0.5316996574401855,
 'r2': 0.8064833283424377,
 'nb': 7.904750823974609,
 'time': 74.41886119171977,
 'num_epochs': 268,
 'best_epoch': 218}

In [8]:
trainer.test_values

{'sample_id': array([  28,  607, 1012,  707,  402, 1015,  874,  399,  110,  905,  339,
         523,  743,  313, 1027,  397,  285,  458,  354,  406,   89,  409,
         178,  249,  978,  158, 1031,  804,  441,  156,  882,  730,  687,
         504,  560,  663,   84,  251,  968,  315,  908,  319, 1160,  565,
        1109,  506,  487,  566, 1005, 1148,  516,  986,   62,  756,   24,
         445,  696,  973,  784,   31,  627,  240,  171,  298, 1128,   22,
         398,  814,  219,  243,  492, 1024,  719,  401,  216,  277,  658,
        1002,    0,  709,   61,  666,  991,  520,  259, 1074,  970, 1110,
         579, 1009,  234,  722,  923,  684,  867,  318,  106,  168, 1004,
        1001,  621,  983,  375,  141,  914,  826,   33,  338, 1038, 1120,
         595,  581, 1156,  797,  678,  657,  550,  927,  735,  751,  529,
         996,  562,  336,  307,  800,  721,  841,  746,  954,  390,  255,
          55,  481,  833,  575,  209,   56,  518,  897,  227, 1170,  852,
         806, 1135,  838,

In [9]:
trainer.test_report

'loss=7.8992    nb=7.9048    mae=0.5317'